
# Parcial Corte 2 — Solución ETL Cafetería

Este notebook evidencia TODO el proceso ETL:
- Extracción
- Transformación (LIMPIEZA EXPLÍCITA)
- Carga a SQLite
- CRUD completo


In [ ]:

import pandas as pd
import sqlite3


## FASE 1 — Carga de datos

In [ ]:

df_prod = pd.read_csv('productos_limpios.csv')
df_cli = pd.read_csv('clientes_limpios.csv')
df_prov = pd.read_csv('proveedores_limpios.csv')

df_prod.head()


## FASE 2 — Limpieza de datos (EVIDENCIA)

In [ ]:

# --- PRODUCTOS ---
# Estandarizar texto
df_prod.columns = df_prod.columns.str.strip().str.lower()

# Asegurar tipos correctos
if 'precio' in df_prod.columns:
    df_prod['precio'] = df_prod['precio'].astype(float)

if 'stock' in df_prod.columns:
    df_prod['stock'] = df_prod['stock'].fillna(0).astype(int)


# --- CLIENTES ---
df_cli.columns = df_cli.columns.str.strip().str.lower()

# Rellenar nulos
df_cli = df_cli.fillna('No Registra')

# Estandarizar texto
for col in df_cli.select_dtypes(include='object').columns:
    df_cli[col] = df_cli[col].str.title()


# --- PROVEEDORES ---
df_prov.columns = df_prov.columns.str.strip().str.lower()

df_prov = df_prov.fillna('No Registra')

for col in df_prov.select_dtypes(include='object').columns:
    df_prov[col] = df_prov[col].str.title()

df_prod.head()


## Exportación

In [ ]:

df_prod.to_csv('parcial_2_productos_limpios.csv', index=False)
df_cli.to_csv('parcial_2_clientes_limpios.csv', index=False)
df_prov.to_csv('parcial_2_proveedores_limpios.csv', index=False)


## FASE 3 — Base de Datos

In [ ]:

conn = sqlite3.connect('parcial_2_cafeteria.db')
cursor = conn.cursor()

df_prod.to_sql('productos', conn, if_exists='replace', index=False)
df_cli.to_sql('clientes', conn, if_exists='replace', index=False)
df_prov.to_sql('proveedores', conn, if_exists='replace', index=False)


## FASE 4 — Tabla ventas

In [ ]:

cursor.execute("""
CREATE TABLE IF NOT EXISTS ventas (
    id_venta INTEGER PRIMARY KEY AUTOINCREMENT,
    id_cliente INTEGER,
    id_producto INTEGER,
    id_proveedor INTEGER,
    cantidad INTEGER,
    total_venta REAL,
    fecha_venta TEXT
);
""")
conn.commit()


## CRUD

In [ ]:

# CREATE
cursor.execute("INSERT INTO ventas VALUES (NULL,1,1,1,2,10000,'2026-04-01')")
cursor.execute("INSERT INTO ventas VALUES (NULL,2,2,1,1,5000,'2026-04-02')")
conn.commit()


In [ ]:

# READ
pd.read_sql("SELECT * FROM ventas", conn)


In [ ]:

# UPDATE
cursor.execute("UPDATE ventas SET cantidad=5 WHERE id_venta=1")
conn.commit()


In [ ]:

# DELETE
cursor.execute("DELETE FROM ventas WHERE id_venta=2")
conn.commit()


In [ ]:
conn.close()